In [5]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from tools.path import setup_paths

paths = setup_paths()
base_dir = os.path.join(paths["output_path"], "processed_dataset")

data_path = os.path.join(base_dir, 'code_rendering')
fix_expertise_path = os.path.join(data_path, 'fix_expertise')
fix_rendering_path = os.path.join(data_path, 'fix_rendering')
both_conditions_path = os.path.join(data_path, 'fix_expertise_rendering')

In [6]:
def visualize_comparison_boxplot(exp: str, info):
    """
    Visualize boxplots comparing a specific metric across different groups for a given trial.
    Enhanced with detailed statistics including median, variance, and outlier counts.
    
    :param: exp: The experiment name or identifier.
    """
    
    if exp == "fix_expertise":
        source_path = fix_expertise_path
    elif exp == "fix_rendering":
        source_path = fix_rendering_path
    else:
        source_path = both_conditions_path
    
    csv_files = [f for f in os.listdir(source_path) if f.endswith(".csv")]
    
    graph_folder = os.path.join(paths["output_path"], "exp_graph", "code_rendering")
    group_folder = os.path.join(graph_folder, exp)
    boxplot_folder = os.path.join(group_folder, 'boxplot')
    os.makedirs(boxplot_folder, exist_ok=True)
    
    dimensions = ["Shape", "Direction", "Length", "Position", "Duration"]
    
    # Make a boxplot for each dimension
    for metric_column in dimensions:
        fig, ax = plt.subplots(figsize=(16, 10))
        
        # Collect data from all CSV files
        all_data = []
        labels = []
        sample_counts = []
        
        for csv_file in csv_files:
            csv_path = os.path.join(source_path, csv_file)
            df = pd.read_csv(csv_path)
            
            # Check if the dimension column exists
            if metric_column in df.columns:
                # Get the data for that dimension
                data = df[metric_column].dropna()
                all_data.append(data)
                sample_counts.append(len(data))
                
                # Use the filename as the label (remove .csv suffix)
                label = csv_file.replace('_result.csv', '').replace('.csv', '')
                labels.append(label)
        
        # Create boxplots
        if all_data:
            bp = ax.boxplot(all_data, tick_labels=labels, patch_artist=True, 
                           showfliers=True,
                           medianprops=dict(color='black', linewidth=2),
                           boxprops=dict(linewidth=1.5),
                           whiskerprops=dict(linewidth=1.5),
                           capprops=dict(linewidth=1.5),
                           flierprops=dict(marker='o', markerfacecolor='white', 
                                         markeredgecolor='black', markersize=6, alpha=0.5))
            
            colors = ['#E74C3C', '#3498DB', '#F39C12', '#2ECC71', '#9B59B6']
            if len(all_data) > len(colors):
                colors = plt.cm.Set2(np.linspace(0, 1, len(all_data)))
            
            for patch, color in zip(bp['boxes'], colors[:len(all_data)]):
                patch.set_facecolor(color)
                patch.set_edgecolor('black')
                patch.set_linewidth(1.5)
                patch.set_alpha(0.8)
            
            # Add median labels and outlier statistics
            y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
            outlier_labels = []  # Store outlier labels
            
            for i, (data, label) in enumerate(zip(all_data, labels)):
                # Calculate statistics
                median = np.median(data)
                q1 = np.percentile(data, 25)
                q3 = np.percentile(data, 75)
                iqr = q3 - q1
                lower_bound = q1 - 1.5 * iqr
                upper_bound = q3 + 1.5 * iqr
                
                # Calculate outliers
                outliers = data[(data < lower_bound) | (data > upper_bound)]
                n_outliers = len(outliers)
                outlier_pct = (n_outliers / len(data)) * 100 if len(data) > 0 else 0
                
                # Add median label at the center of the box
                ax.text(i + 1, median, f'Med: {median:.3f}', 
                       ha='center', va='center', fontsize=10, 
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='black', linewidth=1))
                
                # Store outlier information
                if n_outliers > 0:
                    outlier_labels.append(f'Outliers: {n_outliers}/{len(data)} ({outlier_pct:.1f}%)')
                else:
                    outlier_labels.append('')
            
            # Set chart title and labels
            title = f'Distribution of {metric_column} across Groups - {exp}'
            ax.set_title(title, fontsize=18, fontweight='bold', pad=20)
            ax.set_ylabel(f'{metric_column} Value', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.2, axis='y', linestyle='--')
            
            # Set X-axis labels including sample counts
            x_labels = []
            for label, count in zip(labels, sample_counts):
                x_labels.append(f'{label}\nn: {count}')
            
            ax.set_xticklabels(x_labels, fontsize=11)
            
            # Add red outlier statistics below X-axis labels
            for i, outlier_label in enumerate(outlier_labels):
                if outlier_label:
                    ax.text(i + 1, -0.08, outlier_label, 
                           ha='center', va='top', fontsize=9, 
                           color='red', fontweight='bold',
                           transform=ax.get_xaxis_transform())
            
            plt.subplots_adjust(top=0.93, bottom=0.1)
            
            output_file = os.path.join(boxplot_folder, f'{metric_column}_comparison.png')
            plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
            plt.close()
            
            if info:
                print(f'Saved {metric_column} boxplot to {output_file}')
        else:
            print(f'No data found for {metric_column}')

In [7]:
def visualize_comparison_histogram(exp: str, metric_column: str, bins: int = 30, info: bool = False):
    """
    Overlay histograms comparing a specific dimension across groups and save the plot.
    
    :param: exp: Experiment name 
    :param: metric_column: Dimension column name
    :param: bins: Number of bins for the histogram
    :param: info: Whether to print save information
    """
    if exp == "fix_expertise":
        source_path = fix_expertise_path
    elif exp == "fix_rendering":
        source_path = fix_rendering_path
    else:
        source_path = both_conditions_path

    csv_files = [f for f in os.listdir(source_path) if f.endswith(".csv")]
    graph_folder = os.path.join(paths["output_path"], "exp_graph", "code_rendering")
    group_folder = os.path.join(graph_folder, exp)
    hist_folder = os.path.join(group_folder, "histogram_overlay")
    os.makedirs(hist_folder, exist_ok=True)

    all_values = []
    all_groups = []

    for csv_file in csv_files:
        df = pd.read_csv(os.path.join(source_path, csv_file))
        if metric_column in df.columns:
            series = df[metric_column].dropna()
            if not series.empty:
                group_name = csv_file.replace("_result.csv", "").replace(".csv", "")
                all_values.extend(series.tolist())
                all_groups.extend([group_name] * len(series))

    if not all_values:
        print(f"No data found for {metric_column} in {exp}")
        return

    plot_df = pd.DataFrame({"Group": all_groups, metric_column: all_values})
    plt.figure(figsize=(12, 7))

    sns.histplot(
        data=plot_df,
        x=metric_column,
        hue="Group",
        bins=bins,
        alpha=0.55,
        kde=True,
        stat="density",
        edgecolor="black"
    )

    legend_labels = []
    for group in plot_df["Group"].unique():
        gvals = plot_df[plot_df["Group"] == group][metric_column]
        legend_labels.append(f"{group} (n={len(gvals)}, μ={gvals.mean():.3f}, σ={gvals.std():.3f})")

    plt.title(f"Distribution of {metric_column} across Groups - {exp}", fontsize=16, fontweight="bold")
    plt.xlabel(metric_column, fontsize=12, fontweight="bold")
    plt.ylabel("Density", fontsize=12, fontweight="bold")
    plt.legend(labels=legend_labels, title="Groups", title_fontsize=11, fontsize=10, loc="upper right")
    plt.grid(True, alpha=0.25, linestyle="--")
    plt.tight_layout()

    out_file = os.path.join(hist_folder, f"{metric_column}_comparison_hist.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    if info:
        print(f"Saved histogram overlay to {out_file}")

def visualize_comparison_histogram_subplots(exp: str, metric_column: str, bins: int = 20, info: bool = False):
    """
    Overlay histograms comparing a specific dimension across groups and save the plot.
    
    :param: exp: Experiment name
    :param: metric_column: Dimension column name
    :param: bins: Number of bins for the histogram
    :param: info: Whether to print save information
    """
    if exp == "within_expertise":
        source_path = fix_expertise_path
    elif exp == "within_rendering":
        source_path = fix_rendering_path
    else:
        source_path = both_conditions_path

    csv_files = [f for f in os.listdir(source_path) if f.endswith(".csv")]
    graph_folder = os.path.join(paths["output_path"], "exp_graph", "code_rendering")
    group_folder = os.path.join(graph_folder, exp)
    hist_folder = os.path.join(group_folder, "histogram_subplots")
    os.makedirs(hist_folder, exist_ok=True)

    group_data = {}
    for csv_file in csv_files:
        df = pd.read_csv(os.path.join(source_path, csv_file))
        if metric_column in df.columns:
            series = df[metric_column].dropna()
            if not series.empty:
                group_name = csv_file.replace("_result.csv", "").replace(".csv", "")
                group_data[group_name] = series.tolist()

    if not group_data:
        print(f"No data found for {metric_column} in {exp}")
        return

    n_groups = len(group_data)
    n_cols = min(3, n_groups)
    n_rows = (n_groups + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    axes = axes.flatten() if n_groups > 1 else [axes]

    colors = plt.cm.Set2(np.linspace(0, 1, n_groups))

    for i, (group_name, data) in enumerate(group_data.items()):
        ax = axes[i]
        ax.hist(data, bins=bins, density=True, alpha=0.75, edgecolor="black", color=colors[i])
        mean_val = np.mean(data)
        std_val = np.std(data)
        median_val = np.median(data)
        count = len(data)

        ax.axvline(mean_val, color="red", linestyle="--", linewidth=2, label=f"Mean: {mean_val:.3f}")
        ax.axvline(median_val, color="blue", linestyle="-", linewidth=2, label=f"Median: {median_val:.3f}")
        ax.set_title(f"{group_name}\n(n={count}, σ={std_val:.3f})", fontweight="bold")
        ax.set_xlabel(metric_column, fontweight="bold")
        ax.set_ylabel("Density", fontweight="bold")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.25)

    for j in range(len(group_data), len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(f"Distribution of {metric_column} by Group - {exp}", fontsize=16, fontweight="bold", y=1.02)
    plt.tight_layout()

    out_file = os.path.join(hist_folder, f"{metric_column}_hist_subplots.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()
    if info:
        print(f"Saved histogram subplots to {out_file}")


In [8]:
exps = ['fix_expertise', 'fix_rendering', 'fix_expertise_rendering']
dimensions = ["Shape", "Direction", "Length", "Position", "Duration"]

for exp in exps:
    visualize_comparison_boxplot(exp=exp, info=False)
    for dim in dimensions:
        visualize_comparison_histogram(exp=exp, metric_column=dim, bins=30, info=False)
        visualize_comparison_histogram_subplots(exp=exp, metric_column=dim, bins=20, info=False)
    print(f'Completed visualizations for experiment: {exp}')
    

Completed visualizations for experiment: fix_expertise
Completed visualizations for experiment: fix_rendering
Completed visualizations for experiment: fix_expertise_rendering
